# Cell marker positivity — inference

Runs a trained checkpoint over every cell and every marker channel of one region and
writes a positivity volume, using the value convention of the training masks:
0 background, 1 negative, 2 positive.

The crop handed to the model is built exactly as in `TrainModel.ipynb`, and how many
channels to build is read from the checkpoint rather than set by hand, so the two can
never fall out of step.

In [ ]:
import numpy as np
import nibabel as nib
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy import ndimage

MARKERS_PATH = "EMMA2.nii"
MASK_PATH = "EMMA2_mask.nii"
OUT_PATH = "EMMA2_inference.nii"
MODEL_PATH = "checkpoints/marker_cnn_epoch_200.h5"

CROP = 64
THRESHOLD = 0.5        # probability above which a cell is called positive
BATCH_SIZE = 512
CHUNK = 2048           # cells prepared at a time
DETAIL_CHANNEL = 0     # channel to show close up at the end

print("TensorFlow:", tf.__version__)

model = tf.keras.models.load_model(MODEL_PATH)

# The checkpoint records how it was trained. The marker channels are an odd number, so
# the channel count alone says whether the mask of the other cells is there as well.
N_CHANNELS = model.input_shape[-1]
USE_OTHER_CELLS = N_CHANNELS % 2 == 1
N_MARKERS = N_CHANNELS - 1 - int(USE_OTHER_CELLS)
N_SIDE = (N_MARKERS - 1) // 2

print(f"Model expects {N_CHANNELS} channels: {N_MARKERS} marker channels "
      f"({N_SIDE} on each side of the target)"
      f"{', the target cell and the other cells' if USE_OTHER_CELLS else ' and the target cell'}")

## Load the region

The segmentation is the same for every marker channel, so the cells are labelled once.
A connected component analysis is what separates them, exactly as during training, which
relies on the cells not touching.

In [ ]:
image = nib.as_closest_canonical(nib.load(MARKERS_PATH))
markers = np.asanyarray(image.dataobj).astype(np.float32)

mask = np.asanyarray(nib.as_closest_canonical(nib.load(MASK_PATH)).dataobj)
if mask.ndim == 3:
    mask = mask[:, :, 0]

if mask.shape != markers.shape[:2]:
    raise ValueError(f"mask is {mask.shape}, markers are {markers.shape[:2]}")

cells, n_cells = ndimage.label(mask > 0)
cells = cells.astype(np.int32)
centres = np.array(ndimage.center_of_mass(
    cells > 0, cells, np.arange(1, n_cells + 1))).astype(np.int32)

print(f"{markers.shape[2]} channels, {n_cells} cells, "
      f"image {markers.shape[0]}x{markers.shape[1]}")

## Running the model

`make_sample` cuts exactly the same crop as `TrainModel.ipynb`: the target channel with
`N_SIDE` neighbours on each side, empty where the stack runs out, then the mask of the
target cell and, when the model was trained with it, the mask of the other cells.
No augmentation is applied here.

In [ ]:
def make_sample(z, cell_id, centre_y, centre_x):
    """The same crop the model was trained on."""
    height, width, n_z = markers.shape
    half = CROP // 2

    y = min(max(centre_y - half, 0), height - CROP)
    x = min(max(centre_x - half, 0), width - CROP)

    # Channels beyond either end of the stack do not exist and stay empty.
    wanted = np.arange(z - N_SIDE, z + N_SIDE + 1)
    exists = (wanted >= 0) & (wanted < n_z)

    planes = np.zeros((CROP, CROP, N_MARKERS), dtype=np.float32)
    planes[..., exists] = markers[y:y + CROP, x:x + CROP, wanted[exists]]

    crop = cells[y:y + CROP, x:x + CROP]
    stack = [planes, (crop == cell_id)[..., None]]
    if USE_OTHER_CELLS:
        stack.append(((crop > 0) & (crop != cell_id))[..., None])

    return np.concatenate(stack, axis=-1).astype(np.float32)


n_z = markers.shape[2]
positivity = np.zeros(markers.shape, dtype=np.uint8)
probability = np.zeros((n_cells, n_z), dtype=np.float32)

for z in range(n_z):
    for start in range(0, n_cells, CHUNK):
        batch_ids = np.arange(start, min(start + CHUNK, n_cells))
        batch = np.stack([make_sample(z, i + 1, *centres[i]) for i in batch_ids])
        probability[batch_ids, z] = model.predict(
            batch, batch_size=BATCH_SIZE, verbose=0).ravel()

    # 0 stays background, 1 is negative and 2 is positive, as in the training masks
    call = np.concatenate([[0], np.where(probability[:, z] >= THRESHOLD, 2, 1)])
    positivity[:, :, z] = call[cells]

    print(f"  channel {z + 1:3d}/{n_z}: {(probability[:, z] >= THRESHOLD).mean():6.1%} positive")

print("\nDone.")

## Save

In [ ]:
nib.save(nib.Nifti1Image(positivity, image.affine), OUT_PATH)
print(f"wrote {OUT_PATH}  {positivity.shape}  {positivity.dtype}")

## Results

In [ ]:
columns = 8
rows = int(np.ceil(n_z / columns))

fig, axes = plt.subplots(rows, columns, figsize=(2.0 * columns, 2.2 * rows))
for z, ax in enumerate(np.atleast_1d(axes).ravel()):
    ax.axis("off")
    if z < n_z:
        ax.imshow(positivity[:, :, z], vmin=0, vmax=2, cmap="viridis", interpolation="nearest")
        ax.set_title(f"z{z}   {(probability[:, z] >= THRESHOLD).mean():.0%}+", fontsize=8)
fig.suptitle("Predicted positivity per channel (dark = negative, bright = positive)")
plt.tight_layout()
plt.show()

z = DETAIL_CHANNEL
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(markers[:, :, z], cmap="gray",
               vmax=max(np.percentile(markers[:, :, z], 99.5), 1e-6))
axes[0].set_title(f"marker channel {z}")
axes[1].imshow(cells > 0, cmap="gray")
axes[1].set_title(f"cell segmentation ({n_cells} cells)")
axes[2].imshow(positivity[:, :, z], vmin=0, vmax=2, cmap="viridis", interpolation="nearest")
axes[2].set_title(f"prediction ({(probability[:, z] >= THRESHOLD).mean():.1%} positive)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()